In [3]:
!pip install gradio pandas numpy


In [4]:
# aqi_gradio_only.py
"""
Gradio-only AQI -> pollutant concentration estimator.
- Single-file Gradio app (no notebook-mode fallback).
- Accepts single AQI or comma/newline-separated AQIs.
- Shows DataFrame of estimates and provides CSV download.
- Deterministic inversion using breakpoint table (same logic as earlier versions).
"""

from typing import List, Dict, Tuple, Union
import math
import tempfile
import os
import pandas as pd
import numpy as np
import gradio as gr

# --- Breakpoints & config ---
AQI_BREAKS = np.array([
    (0.0, 50.0),
    (51.0, 100.0),
    (101.0, 200.0),
    (201.0, 300.0),
    (301.0, 400.0),
    (401.0, 500.0)
])
AQI_CATEGORIES = [
    "Good",
    "Satisfactory",
    "Moderately Polluted",
    "Poor",
    "Very Poor",
    "Severe"
]

POLLUTANT_BREAKPOINTS = {
    "PM10":  [(0,50), (51,100), (101,250), (251,350), (351,430), (430, 10000)],
    "PM2.5": [(0,30), (31,60), (61,90), (91,120), (121,250), (250,10000)],
    "NO2":   [(0,40), (41,80), (81,180), (181,280), (281,400), (400,10000)],
    "O3":    [(0,50), (51,100), (101,168), (169,208), (209,748), (748,10000)],
    "CO":    [(0.0,1.0), (1.1,2.0), (2.1,10.0), (10.0,17.0), (17.0,34.0), (34.0,100.0)],
    "SO2":   [(0,40), (41,80), (81,380), (381,800), (801,1600), (1600,10000)],
    "NH3":   [(0,200), (201,400), (401,800), (801,1200), (1200,1800), (1800,10000)],
    "Pb":    [(0.0,0.5), (0.5,1.0), (1.1,2.0), (2.1,3.0), (3.1,3.5), (3.5,100.0)]
}

UNITS = {
    "PM10": "µg/m³", "PM2.5": "µg/m³", "NO2": "µg/m³", "O3": "µg/m³",
    "CO": "mg/m³", "SO2": "µg/m³", "NH3": "µg/m³", "Pb": "µg/m³"
}

# Precompute arrays for speed
_pollutants = list(POLLUTANT_BREAKPOINTS.keys())
_num_intervals = AQI_BREAKS.shape[0]
_pollutant_C_low = np.zeros((len(_pollutants), _num_intervals), dtype=float)
_pollutant_C_high = np.zeros_like(_pollutant_C_low)
for pi, p in enumerate(_pollutants):
    arr = np.array(POLLUTANT_BREAKPOINTS[p], dtype=float)
    _pollutant_C_low[pi, :] = arr[:, 0]
    _pollutant_C_high[pi, :] = arr[:, 1]

_AQI_I_low = AQI_BREAKS[:, 0].astype(float)
_AQI_I_high = AQI_BREAKS[:, 1].astype(float)

_slope = np.zeros_like(_pollutant_C_low)
for pi in range(len(_pollutants)):
    for j in range(_num_intervals):
        C_low = _pollutant_C_low[pi, j]
        C_high = _pollutant_C_high[pi, j]
        I_low = _AQI_I_low[j]
        I_high = _AQI_I_high[j]
        if C_high == C_low:
            _slope[pi, j] = math.inf
        else:
            _slope[pi, j] = (I_high - I_low) / (C_high - C_low)

# --- Core logic ---
def aqi_category(aqi: float) -> str:
    try:
        a = float(aqi)
    except:
        return "Invalid"
    if a < 0:
        return "Invalid (AQI < 0)"
    for (low, high), cat in zip(AQI_BREAKS.tolist(), AQI_CATEGORIES):
        if low <= a <= high:
            return cat
    if a > AQI_BREAKS[-1,1]:
        return "Hazardous (AQI > 500)"
    return "Unknown"

def invert_single_aqi(aqi: float) -> Dict[str, Tuple[float, Tuple[float,float]]]:
    """Return pollutant -> (concentration_estimate, (Clow, Chigh)) for one AQI."""
    if aqi is None:
        return {p: (None, (None, None)) for p in _pollutants}
    aqi = float(aqi)
    # locate interval index
    idx = None
    for i, (Il, Ih) in enumerate(zip(_AQI_I_low, _AQI_I_high)):
        if Il <= aqi <= Ih:
            idx = i
            break
    if idx is None:
        idx = _num_intervals - 1  # extrapolate above final interval

    res = {}
    for pi, p in enumerate(_pollutants):
        C_low = float(_pollutant_C_low[pi, idx])
        C_high = float(_pollutant_C_high[pi, idx])
        s = float(_slope[pi, idx])
        if math.isinf(s):
            conc = C_low
        else:
            conc = ((aqi - _AQI_I_low[idx]) / s) + C_low
            # clamp inside interval if within AQI interval
            if aqi <= _AQI_I_high[idx]:
                conc = min(max(conc, C_low), C_high)
        res[p] = (round(float(conc), 6), (C_low, C_high))
    return res

def invert_batch(aqi_inputs: Union[str, List[float], float]) -> pd.DataFrame:
    """
    Accepts a single float, a list of floats, or a string like "25, 150\n300".
    Returns DataFrame and path to temporary CSV (for download).
    """
    # parse aqi_inputs into list of floats
    if aqi_inputs is None:
        raise ValueError("No AQI provided.")
    if isinstance(aqi_inputs, str):
        parts = [p.strip() for p in aqi_inputs.replace("\n", ",").split(",") if p.strip() != ""]
        try:
            aqi_list = [float(x) for x in parts]
        except:
            raise ValueError("Could not parse input. Provide number(s) separated by comma/newline.")
    elif isinstance(aqi_inputs, (list, tuple, np.ndarray)):
        aqi_list = [float(x) for x in aqi_inputs]
    else:
        aqi_list = [float(aqi_inputs)]

    rows = []
    for aqi in aqi_list:
        inv = invert_single_aqi(aqi)
        row = {"AQI": aqi, "Category": aqi_category(aqi)}
        for p in _pollutants:
            conc, (Clow, Chigh) = inv[p]
            unit = UNITS.get(p, "")
            row[p] = f"{conc} {unit}"
            row[f"{p}_num"] = conc
            row[f"{p}_bp"] = f"{Clow}–{Chigh}"
        rows.append(row)
    df = pd.DataFrame(rows)
    # save CSV to a temporary file and return path for download
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".csv", prefix="aqi_pred_")
    tmp.close()
    df.to_csv(tmp.name, index=False)
    return df, tmp.name

# --- Gradio UI ---
def gradio_predict(aqi_input):
    try:
        df, csv_path = invert_batch(aqi_input)
    except Exception as e:
        # Return empty table and error message
        return pd.DataFrame(), f"Error: {e}", None, (
            "Each pollutant estimate assumes that pollutant alone caused the AQI. "
            "AQI alone does not uniquely determine full pollutant concentrations."
        )
    # Prepare DataFrame for display (human-friendly)
    disp_cols = ["AQI", "Category"] + _pollutants + [f"{p}_bp" for p in _pollutants]
    df_disp = df[disp_cols]
    message = f"Processed {len(df_disp)} row(s)."
    note = (
        "Each pollutant's concentration is the value that WOULD produce the given AQI if that pollutant alone were dominant. "
        "AQI alone cannot uniquely determine all pollutant concentrations; add time/location for better results."
    )
    return df_disp, message, csv_path, note

with gr.Blocks(title="AQI → Pollutant Concentrations (Gradio)") as app:
    gr.Markdown(
        "### AQI → pollutant concentration estimator (deterministic)\n"
        "Paste a single AQI (e.g. `150`) or multiple AQIs separated by commas/newlines (e.g. `25, 150\\n300`).\n\n"
        "**Important:** These are *candidate* concentrations — AQI alone is not a unique inversion."
    )
    with gr.Row():
        aqi_input = gr.Textbox(label="AQI (single or comma/newline-separated)", placeholder="e.g. 75 or 25, 150, 275", lines=2)
        run_btn = gr.Button("Estimate")
    with gr.Row():
        result_table = gr.Dataframe(headers=None, interactive=False, label="Estimated pollutant concentrations")
    with gr.Row():
        status = gr.Markdown()
    with gr.Row():
        csv_file = gr.File(label="Download CSV (latest results)")
    note_md = gr.Markdown()
    run_btn.click(fn=gradio_predict, inputs=[aqi_input], outputs=[result_table, status, csv_file, note_md])

if __name__ == "__main__":
    # Launch. Set share=True if you want a public link (Gradio will try to create one).
    app.launch(share=True)


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://27dda8c87c68146bca.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
